In [ ]:
import matplotlib.pyplot as plt

import numpy as np

import config, plotting
from dimelo import parse_bam, plot_enrichment_profile, plot_reads, load_processed

In [ ]:
reference_file = config.raw_data_dir / "chm13v2.0.fa"
ctcf_bam_file = config.processed_data_dir / "20240923_dimelo_access_background.dorado_v102_sup_v500_5mC_5hmC_v3_6mA_v3.trim.align.filtered.q10.bam"
ctcf_bed_file = config.processed_data_dir / "gm12878_ctcf_chip_intersect_motif.top3k.bed"
ctcf_output_id = f"{ctcf_bam_file.stem}.{ctcf_bed_file.stem}"

ctcf_window_size = 1000
motifs = ["A,0", "WCG,1", "GCH,1"]
mod_prob_thresh = 0.95
n_cores = 10

In [ ]:
parse_bam.pileup(
    input_file=ctcf_bam_file,
    output_name=f"pileup.{ctcf_output_id}",
    ref_genome=reference_file,
    output_directory=config.processed_data_dir,
    regions=ctcf_bed_file,
    motifs=motifs,
    thresh=mod_prob_thresh,
    window_size=ctcf_window_size,
    cores=n_cores,
    override_checks=True
)

parse_bam.extract(
    input_file=ctcf_bam_file,
    output_name=f"extract.{ctcf_output_id}",
    ref_genome=reference_file,
    output_directory=config.processed_data_dir,
    regions=ctcf_bed_file,
    motifs=motifs,
    thresh=None, # No thresholding at extract time
    window_size=ctcf_window_size,
    cores=n_cores,
    override_checks=True,
)

In [ ]:
plot_enrichment_profile.by_modification(
    config.processed_data_dir / f"pileup.{ctcf_output_id}" / "pileup.sorted.bed.gz",
    regions=ctcf_bed_file,
    motifs=motifs,
    window_size=ctcf_window_size,
    smooth_window=25,
    palette=plotting.default_palette_map
)
plt.show()
plt.close()

In [ ]:
_, axs = plt.subplots(1, 3, figsize=(15, 10), layout="constrained")
# TODO: This is currently necessary because extract doesn't allow for auto-thresholding
threshs = [0.6816406, 0.6640625, 0.6640625]
for motif, thresh, ax in zip(motifs, threshs, axs):
    plot_reads.plot_reads(
        mod_file_name=config.processed_data_dir / f"extract.{ctcf_output_id}" / "reads.combined_basemods.h5",
        regions=ctcf_bed_file,
        motifs=motif,
        window_size=ctcf_window_size,
        regions_5to3prime=False,
        sort_by="read_start",
        thresh=mod_prob_thresh,
        ax=ax,
        palette=plotting.default_palette_map
    )
    ax.set_title(motif)
plt.show()
plt.close()

In [ ]:
motif_mod_vector_dict = {}
for motif in motifs:
    read_tuples, entry_labels, _ = load_processed.read_vectors_from_hdf5(
        file=config.processed_data_dir / f"extract.{ctcf_output_id}" / "reads.combined_basemods.h5",
        motifs=[motif],
        regions=ctcf_bed_file,
        window_size=ctcf_window_size,
        single_strand=False,
        sort_by="shuffle",
        calculate_mod_fractions=False,
    )
    mod_vector_index = entry_labels.index("mod_vector")
    val_vector_index = entry_labels.index("val_vector")
    motif_mod_vector_dict[motif] = np.concatenate([read_tuple[mod_vector_index][read_tuple[val_vector_index] == 1] for read_tuple in read_tuples])